# CLUSTERS_CHECK: the earlier TF–IDF and k-means step

Supplementary check, reported in the response to the reviewers rather than in the paper. The originally submitted manuscript grouped fault messages by TF–IDF vectors and k-means clustering (k = 20) and discarded five clusters. This notebook

1. reproduces that clustering (TF–IDF with English stop words, k-means with k = 20 and seed 42);
2. lists every message in the five discarded clusters within the logging period;
3. maps the 15 retained clusters to categories, with their leading TF–IDF terms and episode counts;
4. computes Shannon entropy over the episodes for k = 10, 15, 20, 25 and 30.

It runs on an intermediate version of the fault log (suffix `_v3`), on which the check was made. The paper replaces the clustering with the explicit exclusion rule of `02_REBUILD`.

Inputs: `../ANALYSIS1/DATA/37011/37011FL.csv`, `DATA/37011FLv3.csv`, `build/fault_episodes_v3.csv`. Outputs: `OUTPUT/v3/rev_clusters_*.csv`.

In [ ]:
import os, sys, re
import numpy as np, pandas as pd, sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

env = os.environ.get
CFG = dict(
    raw_log   = "../ANALYSIS1/DATA/37011/37011FL.csv",
    flv1      = "DATA/37011FLv3.csv",
    episodes  = "build/fault_episodes_v3.csv",
    outdir    = env("REV_OUTDIR", "OUTPUT") + "/v3",
    shed      = "ELECTRIC LOCO SHED TATA NAGAR",
    k = 20, seed = 42, removed = [1, 4, 8, 16, 18],
    k_grid = [10, 15, 20, 25, 30],
    blower = "AUX CAP MACH RM BLOW1 NOT OFF",
)
CATEGORY = {0: "Ventilation", 2: "Protection system", 3: "Protection system",
            5: "Traction and power conversion", 6: "Traction and power conversion",
            7: "Traction and power conversion", 9: "Braking and pneumatic",
            10: "Protection system", 11: "Protection system",
            12: "Traction and power conversion", 13: "Protection system",
            14: "Traction and power conversion", 15: "Auxiliary",
            17: "Power supply quality", 19: "Power supply quality"}   # 37011FLGEO cell 38
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 1. Reproduce the original clustering

The corpus is built as in the original analysis: shed records removed, the description split at the colon, and the message text clustered, recovery messages included.

In [ ]:
raw = pd.read_csv(CFG["raw_log"])
raw1 = raw[raw["LOCATION"] != CFG["shed"]].reset_index(drop=True)
t0 = raw1["DESCRIPTION"].str.split(":")
raw1 = raw1.assign(FAULT=t0.apply(lambda x: x[1]), T1=t0.apply(lambda x: x[0]))
raw1["SYSTEMPART"] = raw1["T1"].str.lstrip("[FP0123456789NODIS] ")
raw1 = raw1.drop(columns=["DESCRIPTION", "T1"])
print(f"{len(raw):,} raw records; {len(raw1):,} after removing the shed location")

def cluster(k, texts=None):
    tf = TfidfVectorizer(stop_words="english")
    X = tf.fit_transform(raw1["FAULT"])
    km = KMeans(n_clusters=k, random_state=CFG["seed"]).fit(X)
    return tf, km, km.labels_

TF, KM, lab = cluster(CFG["k"])
raw1["CLUSTER"] = lab
raw2 = raw1[~raw1["CLUSTER"].isin(CFG["removed"])].reset_index(drop=True)

flv1 = pd.read_csv(CFG["flv1"])
head = raw2.head(len(flv1))
same_rows = head["DATETIME"].equals(flv1["DATETIME"]) and head["FAULT"].equals(flv1["FAULT"])
same_lab = (head["CLUSTER"].values == flv1["CLUSTER"].values).mean()
print(f"First {len(flv1):,} retained records match 37011FLv3 row for row: {same_rows}")
print(f"Cluster labels identical to 37011FLv3: {100*same_lab:.2f}%")
if not env("REV_TEST"):
    assert same_rows and same_lab == 1.0, "original clustering not reproduced; stop and investigate"

## 2. What the five discarded clusters contained

Records in the logging period (from the first record of `37011FLv3.csv`) that fell in the
discarded clusters, by message, with onsets and recovery messages counted separately.

In [ ]:
raw1["dt"] = pd.to_datetime(raw1["DATETIME"], format="%m/%d/%Y %H:%M:%S")
start = pd.to_datetime(flv1["DATETIME"], format="%m/%d/%Y %H:%M:%S").min()
win = raw1[raw1.dt >= start].copy()
win["is_rec"] = win["FAULT"].str.contains("Recovered", case=False)
win["base"] = win["FAULT"].str.replace(r"\s*recovered\s*$", "", flags=re.I, regex=True).str.strip()
rem = (win[win.CLUSTER.isin(CFG["removed"])]
         .groupby(["CLUSTER", "base"]).agg(onsets=("is_rec", lambda s: int((~s).sum())),
                                           recoveries=("is_rec", "sum"))
         .reset_index().sort_values(["CLUSTER", "onsets"], ascending=[True, False]))
print(f"Logging period from {start}: {len(win):,} records, of which "
      f"{int(win.CLUSTER.isin(CFG['removed']).sum()):,} in discarded clusters "
      f"({int((win.CLUSTER.isin(CFG['removed']) & ~win.is_rec).sum()):,} onsets)\n")
print(rem.to_string(index=False))
rem.to_csv(f"{CFG['outdir']}/rev_clusters_removed.csv", index=False)

ep = pd.read_csv(CFG["episodes"], parse_dates=["onset"])
gone = set(rem.base.str.upper())
print(f"\nDiscarded messages present among the 4,521 episodes: {len(gone & set(ep.fault))}")
print(f"Retained records in the period: {int((~win.CLUSTER.isin(CFG['removed'])).sum()):,} (37011FLv3 has {len(flv1):,})")

## 3. Mapping table

For each retained cluster: its category, its distinct messages, the leading terms of its TF–IDF centroid, and its share of the episodes. Each message is checked to belong to one cluster only.

In [ ]:
terms = np.array(TF.get_feature_names_out())
rows = []
for c in sorted(CATEGORY):
    msgs = sorted(win.loc[(win.CLUSTER == c) & ~win.is_rec, "base"].unique())
    top = terms[np.argsort(KM.cluster_centers_[c])[::-1][:4]]
    n = int((ep.cluster == c).sum())
    rows.append(dict(cluster=c, category=CATEGORY[c], labels=len(msgs),
                     leading_terms=", ".join(top), episodes=n,
                     share_pct=round(100 * n / len(ep), 1), messages="; ".join(msgs)))
mapping = pd.DataFrame(rows)
print(mapping.drop(columns="messages").to_string(index=False))
print(f"\nEpisodes covered: {mapping.episodes.sum():,} of {len(ep):,}")
per_label = ep.groupby("fault")["cluster"].nunique()
print(f"Labels assigned to more than one cluster: {int((per_label > 1).sum())} of {len(per_label)}")
cat = (ep.assign(category=ep.cluster.map(CATEGORY)).groupby("category").size()
         .sort_values(ascending=False).rename("episodes"))
print("\nEpisodes by Table 1 category\n", cat.to_string())
mapping.to_csv(f"{CFG['outdir']}/rev_clusters_mapping.csv", index=False)

## 4. Entropy and the choice of k

For each k the corpus is re-clustered with the same settings, and each episode is assigned the cluster of its message. Shannon entropy is computed over the clusters that episodes occupy, with the maximum log2 of that number. The message-level and category-level values do not depend on k. Each is also given without the blower message.

In [ ]:
def shannon(counts):
    p = np.asarray(counts, float); p = p[p > 0] / p.sum()
    h = float(-(p * np.log2(p)).sum()); hmax = float(np.log2(len(p)))
    return h, hmax, h / hmax if hmax else np.nan, len(p)

text_ep = ep.fault.str.title()                     # case does not matter to TF-IDF
blower = ep.fault.eq(CFG["blower"]).values
rows = []
for k in CFG["k_grid"]:
    tf, km, _ = cluster(k)
    assigned = km.predict(tf.transform(text_ep))
    if k == CFG["k"]:
        agree = (assigned == ep.cluster.values).mean()
        print(f"k = 20: message-based assignment agrees with the stored cluster for {100*agree:.2f}% of episodes")
    for subset, m in [("all labels", np.ones(len(ep), bool)), ("without blower", ~blower)]:
        h, hmax, hn, kk = shannon(pd.Series(assigned[m]).value_counts())
        rows.append(dict(measure=f"cluster, k = {k}", subset=subset, H_bits=h, Hmax_bits=hmax, normalised=hn, categories=kk))
for name, col in [("fault label", ep.fault), ("Table 1 category", ep.cluster.map(CATEGORY))]:
    for subset, m in [("all labels", np.ones(len(ep), bool)), ("without blower", ~blower)]:
        h, hmax, hn, kk = shannon(col[m].value_counts())
        rows.append(dict(measure=name, subset=subset, H_bits=h, Hmax_bits=hmax, normalised=hn, categories=kk))
ent = pd.DataFrame(rows)
print(ent.pivot(index="measure", columns="subset", values=["H_bits", "Hmax_bits", "normalised", "categories"]).round(3).to_string())
ent.to_csv(f"{CFG['outdir']}/rev_clusters_entropy.csv", index=False)
print("\nSubmitted Table 14: label 3.873 of 6.492 bits (0.597); cluster 3.012 of 3.907 bits (0.771)")

___
**END**